
##weather to bronze

* Gets data from amazon s3. I upload files incrementally to s3 myself to simulate incremental weather data.

* Source: s3://secret/raw/incremental_load/historic-weather-data/landing

* Destination: himalaya.bronze

#1. Pull Data From S3

In [0]:
%run /Workspace/Repos/nikum.vedansh@gmail.com/himalayan-expeditions-project/0_scripts/configs/credentials

In [0]:
%run /Workspace/Repos/nikum.vedansh@gmail.com/himalayan-expeditions-project/0_scripts/configs/config

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
from io import BytesIO
import pandas as pd
import boto3
import os

In [0]:
s3_client = boto3.client(
    's3',
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ["AWS_DEFAULT_REGION"]
)

In [0]:
LANDING_PREFIX = "raw/incremental_load/historic-weather-data/landing/"
PROCESSED_PREFIX = "raw/incremental_load/historic-weather-data/processed/"

In [0]:
BRONZE_TABLE = "himalaya.bronze.weather"
STAGING_TABLE = "himalaya.bronze.staging_weather"

In [0]:
# List files in landing
response = s3_client.list_objects_v2(Bucket=S3_BUCKET, Prefix=LANDING_PREFIX)
files = [obj["Key"] for obj in response.get("Contents", []) if not obj["Key"].endswith("/")]

print(f"Files found in landing: {len(files)}")
for f in files:
    print(f"  - {f}")

#2. Aggregate all files

In [0]:
dfs = []
for file_key in files:
    # Extract peakid from filename e.g. "AMAD.parquet" → "AMAD"
    peakid = file_key.split("/")[-1].replace(".parquet", "")
    
    # Read parquet from S3 using boto3
    obj = s3_client.get_object(Bucket=S3_BUCKET, Key=file_key)
    pandas_df = pd.read_parquet(BytesIO(obj["Body"].read()))
    
    # Add peakid and ingested_at
    pandas_df["peakid"] = peakid
    pandas_df["ingested_at"] = datetime.now()
    
    dfs.append(pandas_df)

>##Combine in one dataframe

In [0]:
# Combine all into one DataFrame
combined_df = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(combined_df)}")
print(combined_df.dtypes)

>##Convert to Spark DataFrame

In [0]:
spark_df = spark.createDataFrame(combined_df)

>##Append to bronze

In [0]:
spark_df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
print(f"✅ Written to {BRONZE_TABLE}")

>##Overwrite in staging

In [0]:
spark_df.write.format("delta").mode("overwrite").saveAsTable(STAGING_TABLE)
print(f"✅ Written to {STAGING_TABLE}")

#3. Move files from landing to processed

In [0]:
for file_key in files:
    filename = file_key.split("/")[-1]
    processed_key = PROCESSED_PREFIX + filename
    
    # Copy to processed
    s3_client.copy_object(
        Bucket=S3_BUCKET,
        CopySource={"Bucket": S3_BUCKET, "Key": file_key},
        Key=processed_key
    )
    
    # Delete from landing
    s3_client.delete_object(Bucket=S3_BUCKET, Key=file_key)
    
    print(f"✅ Moved {filename} to processed")

#4. Summary

In [0]:
print(f"\n✅ Weather Bronze ingestion complete")
print(f"   Files processed: {len(files)}")
print(f"   Total rows ingested: {combined_df.shape[0]}")
print(f"   Bronze table: {BRONZE_TABLE}")
print(f"   Staging table: {STAGING_TABLE}")

In [0]:
display(spark.table(BRONZE_TABLE).limit(5))

## What This Notebook Does
1. Lists parquet files in S3 landing zone
2. Reads each file using boto3, extracts peakid from filename
3. Combines all peaks into one DataFrame
4. Writes to `himalaya.bronze.weather` (append) — full historical record
5. Writes to `himalaya.bronze.staging_weather` (overwrite) — current batch only
6. Moves processed files from landing to processed